# FastSpeech 2: English (Svarah) & Telugu (IndicTTS) End-to-End Notebook\n**IIIT Hyderabad Voice AI assignment**\n\nOpen this notebook from the `fix/tts-dataset-pipeline` branch. It first runs small real-data smoke tests, then an optional full experiment. It does not fabricate outputs: checkpoints, plots and report results are created only by successful model runs.\n\n**Execution order:** Stage 1 setup and unit tests → Stage 2 small data/alignment/training checks → inspect outputs → manually enable Stage 3 full data and training → synthesis and plots.\n\nUse a Colab GPU runtime (T4 or better) and mount Google Drive. Svarah access may require an accepted Hugging Face token. MFA installation and corpus alignment can take time.

In [ ]:
# 1. GPU Verification & Hardware Audit
!nvidia-smi

import torch
print('PyTorch Version:', torch.__version__)
assert torch.cuda.is_available(), 'ERROR: GPU accelerator not detected! Go to Runtime -> Change runtime type -> Select T4 or A100 GPU.'
gpu_name = torch.cuda.get_device_name(0)
total_vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'\n>>> Active GPU: {gpu_name} ({total_vram_gb:.2f} GB VRAM) <<<')

In [ ]:
# 2. Mount Google Drive for Continuous Persistent Checkpoint Recovery
from google.colab import drive
import os

drive.mount('/content/drive')
drive_root = '/content/drive/MyDrive/fastspeech2_tts'
os.makedirs(f'{drive_root}/English/checkpoints', exist_ok=True)
os.makedirs(f'{drive_root}/Telugu/checkpoints', exist_ok=True)
os.makedirs(f'{drive_root}/Results/English', exist_ok=True)
os.makedirs(f'{drive_root}/Results/Telugu', exist_ok=True)
os.environ['DRIVE_SAVE_DIR'] = drive_root
print(f'Persistent Google Drive workspace initialized at: {drive_root}')

In [ ]:
# 3. Clone the exact repair branch (never silently fall back to main)
import os, subprocess
REPO_URL = "https://github.com/Dhanush1205/fastspeech2-english-telugu-tts.git"
BRANCH = "fix/tts-dataset-pipeline"
REPO_DIR = "/content/fastspeech2-english-telugu-tts"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--single-branch", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
%cd /content/fastspeech2-english-telugu-tts
subprocess.run(["git", "fetch", "origin", BRANCH], check=True)
subprocess.run(["git", "checkout", BRANCH], check=True)
subprocess.run(["git", "pull", "--ff-only", "origin", BRANCH], check=True)
current_branch = subprocess.check_output(["git", "branch", "--show-current"], text=True).strip()
assert current_branch == BRANCH, f"Wrong branch: {current_branch}"
print("Repository:", REPO_URL)
print("Branch:", current_branch)
print("Commit:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())

# Python dependencies for preprocessing, training, logging and report generation.
!pip install -q soundfile librosa pyworld tgt g2p-en transformers datasets pyyaml tqdm matplotlib scipy reportlab tensorboard unidecode

# Keep MFA/Kaldi/Pynini in an isolated Conda environment, separate from Colab PyTorch.
if not os.path.exists("/content/miniforge/bin/conda"):
    subprocess.run(["bash", "-lc", "wget -q https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh -O /tmp/Miniforge3.sh && bash /tmp/Miniforge3.sh -b -p /content/miniforge"], check=True)
if not os.path.exists("/content/mfa_env/bin/mfa"):
    subprocess.run(["/content/miniforge/bin/conda", "create", "-y", "-p", "/content/mfa_env", "-c", "conda-forge", "montreal-forced-aligner"], check=True)
subprocess.run(["/content/mfa_env/bin/mfa", "version"], check=True)
subprocess.run(["python", "-c", "import nltk; nltk.download('averaged_perceptron_tagger', quiet=True); nltk.download('averaged_perceptron_tagger_eng', quiet=True); nltk.download('cmudict', quiet=True)"], check=True)
print("Setup completed. Continue only if the MFA version command succeeded.")


In [ ]:
# 4. Run the 13 focused dataset/alignment unit tests used by GitHub Actions
!python -m unittest tests.test_dataset_preparation -v


## STAGE 2: SMOKE TEST BENCHMARKING (Genuine Small Samples)
This stage verifies real dataset audio decoding, separate train/validation manifests, MFA/IndicMFA alignment, duration-to-token mapping, feature extraction, GPU training (100 steps), checkpoint persistence and restore. It does not claim full dataset training.

In [ ]:
# 5. Acquire Genuine Smoke Test Samples (10 Train / 4 Val; utterance-disjoint, speaker split unverified)
import os
from getpass import getpass

# Svarah is a gated Hugging Face dataset. If not already set in environment, prompt securely:
if 'HF_TOKEN' not in os.environ:
    token = getpass('Enter your Hugging Face Access Token (HF_TOKEN) for Svarah [hidden]: ')
    if token.strip():
        os.environ['HF_TOKEN'] = token.strip()

print('--- Downloading English (Svarah) Smoke Test Samples ---')
!python data_prep/prepare_svarah.py \
    --smoke_test \
    --output_dir data/svarah_smoke \
    --splits_dir English/data_splits

print('\n--- Downloading Telugu (IndicTTS) Smoke Test Samples ---')
!python data_prep/prepare_indictts.py \
    --smoke_test \
    --output_dir data/indictts_smoke \
    --splits_dir Telugu/data_splits

# Verify Manifests and Smoke Test Integrity
print('\n' + '=' * 70)
print('VERIFYING SMOKE TEST INTEGRITY & ZERO LEAKAGE')
print('=' * 70)
for lang, split_dir in [('English', 'English/data_splits'), ('Telugu', 'Telugu/data_splits')]:
    train_lines = [l.strip().split('|') for l in open(f'{split_dir}/train.txt', encoding='utf-8') if l.strip()]
    val_lines = [l.strip().split('|') for l in open(f'{split_dir}/val.txt', encoding='utf-8') if l.strip()]
    train_ids = {l[0] for l in train_lines}
    val_ids = {l[0] for l in val_lines}
    id_leakage = train_ids.intersection(val_ids)
    with open(f'{split_dir}/split_metadata.json', encoding='utf-8') as f:
        metadata = json.load(f)
    print(f'[{lang}] Train Samples: {len(train_lines)} | Val Samples: {len(val_lines)}')
    print(f'[{lang}] Utterance Leakage: {len(id_leakage)} (Passed: {len(id_leakage) == 0})')
    speaker_status = metadata.get('speaker_disjoint')
    if speaker_status is True:
        print(f'[{lang}] Speaker-disjoint: verified by source speaker IDs')
    else:
        print(f'[{lang}] Speaker-disjoint: NOT VERIFIED (source has no reliable unique speaker ID)')
    assert len(train_lines) == 10, f'Expected 10 train lines, got {len(train_lines)}'
    assert len(val_lines) == 4, f'Expected 4 val lines, got {len(val_lines)}'
    assert len(id_leakage) == 0, f'Utterance ID leakage detected: {id_leakage}'
print('=' * 70)
print('ALL SMOKE TEST SAMPLES & MANIFESTS VERIFIED WITH ZERO LEAKAGE!')

In [ ]:
# 6. Download Official AI4Bharat IndicMFA Telugu Acoustic Model & Dictionary
!python Telugu/preprocess.py --download_indicmfa_models

In [ ]:
# 7. Create MFA corpora and generate real TextGrid alignments for smoke samples.
# Merge both train and validation manifests so each utterance receives duration supervision.
!mkdir -p data/alignment/smoke
!cat English/data_splits/train.txt English/data_splits/val.txt > data/alignment/smoke/english_all.txt
!cat Telugu/data_splits/train.txt Telugu/data_splits/val.txt > data/alignment/smoke/telugu_all.txt

MFA_BIN = "/content/mfa_env/bin/mfa"

print("--- Aligning English smoke samples with MFA ---")
!python data_prep/align_dataset.py \
    --language english \
    --manifest data/alignment/smoke/english_all.txt \
    --corpus_dir data/alignment/smoke/english_corpus \
    --output_dir data/alignment/smoke/english_textgrids \
    --mfa /content/mfa_env/bin/mfa \
    --jobs 2 \
    --force

print("--- Aligning Telugu smoke samples with IndicMFA ---")
!python data_prep/align_dataset.py \
    --language telugu \
    --manifest data/alignment/smoke/telugu_all.txt \
    --corpus_dir data/alignment/smoke/telugu_corpus \
    --output_dir data/alignment/smoke/telugu_textgrids \
    --mfa /content/mfa_env/bin/mfa \
    --indicmfa_model_dir indicmfa_models \
    --jobs 2 \
    --force

# 8. Preprocess smoke-test features using validated duration alignments.
print("--- Preprocessing English smoke data ---")
!python English/preprocess.py \
    --config English/config/preprocess.yaml \
    --data_dir data/svarah_smoke \
    --output_dir data/preprocessed/english \
    --mfa_dir data/alignment/smoke/english_textgrids

print("--- Preprocessing Telugu smoke data using IndicMFA TextGrids ---")
!python Telugu/preprocess.py \
    --config Telugu/config/preprocess.yaml \
    --data_dir data/indictts_smoke \
    --output_dir data/preprocessed/telugu \
    --mfa_dir data/alignment/smoke/telugu_textgrids


In [ ]:
# 8. Run GPU Training Benchmark & Verify Checkpoint Resumption (100 Steps)
import time
import torch

# A. English Benchmark Training (100 Steps) with Drive Sync
print('=== Starting English GPU Benchmark Training (100 Steps) ===')
t0_en = time.time()
!python English/train.py \
    -p English/config/preprocess.yaml \
    -m English/config/model.yaml \
    -t English/config/train.yaml \
    --total_step 100 \
    --save_step 50 \
    --eval_step 50 \
    --log_step 10 \
    --drive_save_dir /content/drive/MyDrive/fastspeech2_tts/English/checkpoints
en_bench_time = time.time() - t0_en

# Test English Checkpoint Restoration
print('\n=== Testing English Checkpoint Resumption from Step 50 ===')
!python English/train.py \
    -p English/config/preprocess.yaml \
    -m English/config/model.yaml \
    -t English/config/train.yaml \
    --restore_step 50 \
    --total_step 100 \
    --save_step 50 \
    --eval_step 50 \
    --log_step 10 \
    --drive_save_dir /content/drive/MyDrive/fastspeech2_tts/English/checkpoints

# B. Telugu Benchmark Training (100 Steps) with Drive Sync
print('\n=== Starting Telugu GPU Benchmark Training (100 Steps) ===')
t0_te = time.time()
!python Telugu/train.py \
    -p Telugu/config/preprocess.yaml \
    -m Telugu/config/model.yaml \
    -t Telugu/config/train.yaml \
    --total_step 100 \
    --save_step 50 \
    --eval_step 50 \
    --log_step 10 \
    --drive_save_dir /content/drive/MyDrive/fastspeech2_tts/Telugu/checkpoints
te_bench_time = time.time() - t0_te

# Test Telugu Checkpoint Restoration
print('\n=== Testing Telugu Checkpoint Resumption from Step 50 ===')
!python Telugu/train.py \
    -p Telugu/config/preprocess.yaml \
    -m Telugu/config/model.yaml \
    -t Telugu/config/train.yaml \
    --restore_step 50 \
    --total_step 100 \
    --save_step 50 \
    --eval_step 50 \
    --log_step 10 \
    --drive_save_dir /content/drive/MyDrive/fastspeech2_tts/Telugu/checkpoints


In [ ]:
# 9. Verify benchmark outputs without printing assumed success
from pathlib import Path
import numpy as np
import torch

def report_language(lang):
    ckpt_dir = Path(lang) / "checkpoints"
    candidates = sorted(list(ckpt_dir.glob("*.pth.tar")) + list(ckpt_dir.glob("*.pth")))
    loss_file = ckpt_dir / "loss_history.npz"
    print(f"\\n{lang}")
    print("Checkpoint files:", [p.name for p in candidates] or "NONE FOUND")
    if loss_file.exists():
        hist = np.load(loss_file)
        print("Train loss entries:", len(hist.get("train_losses", [])))
        print("Validation loss entries:", len(hist.get("val_losses", [])))
    else:
        print("Loss history: MISSING")
    drive_dir = Path("/content/drive/MyDrive/fastspeech2_tts") / lang / "checkpoints"
    print("Drive checkpoint files:", [p.name for p in drive_dir.glob("*")] if drive_dir.exists() else "DRIVE DIRECTORY MISSING")

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
report_language("English")
report_language("Telugu")
print("\\nDo not proceed to full training unless both smoke pipelines, checkpoint creation and resume tests succeeded.");


## STAGE 3: FULL CONVERGENCE TRAINING & FINAL ARTIFACTS
> [!IMPORTANT]
> **DO NOT RUN THIS STAGE UNTIL THE BENCHMARK RESULTS ARE APPROVED BY THE USER.**
> Once approved, set `PROCEED_WITH_FULL_TRAINING = True` below and execute.

In [ ]:
# 10. Safety Gate for Full Training (User Approval Required)
PROCEED_WITH_FULL_TRAINING = False

if not PROCEED_WITH_FULL_TRAINING:
    raise RuntimeError(
        'Execution halted: Full 200,000-step training requires explicit approval. '
        'Please present the benchmark report from Cell 9 to the user. '
        'Once approved, set PROCEED_WITH_FULL_TRAINING = True to continue.'
    )
print('User approval confirmed! Proceeding with full dataset acquisition and training.')

In [ ]:
# 11. Full Dataset Acquisition (5.0h Train / 1.0h Val per Language)
# A. Svarah Indian English (~5h train / ~1h val)
!python data_prep/prepare_svarah.py \
    --output_dir data/svarah \
    --splits_dir English/data_splits \
    --train_hours 5.0 \
    --val_hours 1.0

# B. IndicTTS Telugu (~5h train / ~1h val)
!python data_prep/prepare_indictts.py \
    --output_dir data/indictts_telugu \
    --splits_dir Telugu/data_splits \
    --train_hours 5.0 \
    --val_hours 1.0


In [ ]:
# 12. Build full train+validation corpora and run forced alignment.
!mkdir -p data/alignment/full
!cat English/data_splits/train.txt English/data_splits/val.txt > data/alignment/full/english_all.txt
!cat Telugu/data_splits/train.txt Telugu/data_splits/val.txt > data/alignment/full/telugu_all.txt

!python data_prep/align_dataset.py \
    --language english \
    --manifest data/alignment/full/english_all.txt \
    --corpus_dir data/alignment/full/english_corpus \
    --output_dir data/alignment/full/english_textgrids \
    --mfa /content/mfa_env/bin/mfa \
    --jobs 4 \
    --force

!python data_prep/align_dataset.py \
    --language telugu \
    --manifest data/alignment/full/telugu_all.txt \
    --corpus_dir data/alignment/full/telugu_corpus \
    --output_dir data/alignment/full/telugu_textgrids \
    --mfa /content/mfa_env/bin/mfa \
    --indicmfa_model_dir indicmfa_models \
    --jobs 4 \
    --force

# 13. Extract aligned mel, pitch, energy and duration features.
!python English/preprocess.py \
    --config English/config/preprocess.yaml \
    --data_dir data/svarah \
    --output_dir data/preprocessed/english \
    --train_hours 5.0 \
    --val_hours 1.0 \
    --mfa_dir data/alignment/full/english_textgrids

!python Telugu/preprocess.py \
    --config Telugu/config/preprocess.yaml \
    --data_dir data/indictts_telugu \
    --output_dir data/preprocessed/telugu \
    --train_hours 5.0 \
    --val_hours 1.0 \
    --mfa_dir data/alignment/full/telugu_textgrids


In [ ]:
# 13. Full FastSpeech 2 Training Loop with Continuous Google Drive Mirroring
# English Model Training
!python English/train.py \
    -p English/config/preprocess.yaml \
    -m English/config/model.yaml \
    -t English/config/train.yaml \
    --drive_save_dir /content/drive/MyDrive/fastspeech2_tts/English/checkpoints

# Telugu Model Training
!python Telugu/train.py \
    -p Telugu/config/preprocess.yaml \
    -m Telugu/config/model.yaml \
    -t Telugu/config/train.yaml \
    --drive_save_dir /content/drive/MyDrive/fastspeech2_tts/Telugu/checkpoints


In [ ]:
# 14. Synthesize Mandatory Test Sentences (E1-E4, T1-T4)
!python English/inference.py --output_dir Results/English/test_generated_audios
!python Telugu/inference.py --output_dir Results/Telugu/test_generated_audios

# Sync all synthesized WAVs to Google Drive
!cp -r Results/English/test_generated_audios/* /content/drive/MyDrive/fastspeech2_tts/Results/English/
!cp -r Results/Telugu/test_generated_audios/* /content/drive/MyDrive/fastspeech2_tts/Results/Telugu/
print('Synthesized audio files copied to Google Drive!')

In [ ]:
# 15. Generate 30 Genuine Variance Adaptor Plots & 4 Loss Curves
!python utils/plot_variance.py
!python utils/plot_losses.py

# Sync plots to Drive
!cp -r Results/English/variance_adaptor_plots /content/drive/MyDrive/fastspeech2_tts/Results/English/
!cp -r Results/Telugu/variance_adaptor_plots /content/drive/MyDrive/fastspeech2_tts/Results/Telugu/
!cp Results/English/*.png /content/drive/MyDrive/fastspeech2_tts/Results/English/
!cp Results/Telugu/*.png /content/drive/MyDrive/fastspeech2_tts/Results/Telugu/

# Recompile Technical Report with Genuine Experimental Results
!python Report/generate_report.py
!cp Report/Technical_Report.pdf /content/drive/MyDrive/fastspeech2_tts/
print('Technical report recompiled and persisted to Google Drive!')

In [ ]:
# 16. In-Notebook Audio Playback
import IPython.display as ipd
print('English E1 Audio (Antidisestablishmentarianism):')
ipd.display(ipd.Audio('Results/English/test_generated_audios/E1.wav'))
print('Telugu T1 Audio (న్యూరోఫార్మకోలాజికల్):')
ipd.display(ipd.Audio('Results/Telugu/test_generated_audios/T1.wav'))